# PAL on Colab


## Mount Google Drive


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Set project paths

In [2]:
from pathlib import Path
import os

DRIVE_ROOT = Path("/content/drive/MyDrive")
PROJECT_ROOT = DRIVE_ROOT / "BT4222" / "PAL"
DATASET_ROOT = PROJECT_ROOT / "datasets"
MODEL_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
LOG_DIR = PROJECT_ROOT / "log"
RUNS_DIR = PROJECT_ROOT / "runs"

for p in [RESULTS_DIR, CHECKPOINT_DIR, LOG_DIR, RUNS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

os.chdir(PROJECT_ROOT)
print("Working directory:", PROJECT_ROOT)
print("models exists:", MODEL_DIR.exists())
print("datasets exists:", DATASET_ROOT.exists())

Working directory: /content/drive/MyDrive/BT4222/PAL
models exists: True
datasets exists: True


## Install dependencies

In [3]:
!pip install -q pyyaml scipy tqdm tensorboard


## Verify required files

In [4]:
required_files = [
    PROJECT_ROOT / "config.yaml",
    PROJECT_ROOT / "utility.py",
    PROJECT_ROOT / "train.py",
    PROJECT_ROOT / "export_explanations.py",
    PROJECT_ROOT / "prepare_steam_pal.py",
    PROJECT_ROOT / "models" / "PAL.py",
]

missing = [str(p) for p in required_files if not p.exists()]
if missing:
    print("Missing files:")
    for p in missing:
        print(" -", p)
else:
    print("All core code files are present.")

All core code files are present.


## Verify SteamDebug dataset files

In [5]:
dataset_name = "SteamDebug"
dataset_dir = DATASET_ROOT / dataset_name

required_dataset_files = [
    dataset_dir / "SteamDebug_data_size.txt",
    dataset_dir / "bundle_item.txt",
    dataset_dir / "user_bundle_train.txt",
    dataset_dir / "user_bundle_tune.txt",
    dataset_dir / "user_bundle_test.txt",
    dataset_dir / "user_item.txt",
]

missing_dataset = [str(p) for p in required_dataset_files if not p.exists()]
if missing_dataset:
    print("Missing dataset files:")
    for p in missing_dataset:
        print(" -", p)
else:
    print("SteamDebug dataset looks complete.")

SteamDebug dataset looks complete.


## Load base configuration

In [6]:
import yaml
import torch

with open(PROJECT_ROOT / "config.yaml", "r", encoding="utf-8") as f:
    full_conf = yaml.safe_load(f)

base_conf = dict(full_conf["SteamDebug"])
base_conf["data_path"] = str(PROJECT_ROOT / "datasets")
base_conf["dataset"] = "SteamDebug"
base_conf["model"] = "PAL"
base_conf["device"] = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("base_conf loaded")
print("keys:", sorted(base_conf.keys()))
print("batch_size_train =", base_conf["batch_size_train"])
print("batch_size_test  =", base_conf["batch_size_test"])
print("topk             =", base_conf["topk"])
print("data_path        =", base_conf["data_path"])
print("device           =", base_conf["device"])

base_conf loaded
keys: ['attention_dropout', 'attention_hidden_size', 'attention_score_type', 'attention_type', 'aug_type', 'batch_size_test', 'batch_size_train', 'bundle_agg_ratios', 'bundle_level_ratios', 'c_lambdas', 'c_temps', 'data_path', 'dataset', 'device', 'ed_interval', 'embedding_sizes', 'epochs', 'eval_bundle_chunk_size', 'explain_top_items', 'fusion_hidden_size', 'fusion_type', 'item_level_ratios', 'l2_regs', 'lrs', 'model', 'neg_num', 'num_layerss', 'num_workers_test', 'num_workers_train', 'test_interval', 'topk', 'use_item_attention', 'use_view_fusion']
batch_size_train = 64
batch_size_test  = 128
topk             = [1, 2, 5, 10, 20]
data_path        = /content/drive/MyDrive/BT4222/PAL/datasets
device           = cuda


## Smoke test: imports, dataset loading, model initialization, and single-batch forward

In [7]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utility import Datasets
from models.PAL import PAL

print("Imports OK")

conf_for_data = dict(base_conf)
dataset = Datasets(conf_for_data)

print("num_users:", dataset.num_users)
print("num_bundles:", dataset.num_bundles)
print("num_items:", dataset.num_items)
print("train batches:", len(dataset.train_loader))
print("val batches:", len(dataset.val_loader))
print("test batches:", len(dataset.test_loader))

conf_for_model = dict(conf_for_data)
conf_for_model["num_users"] = dataset.num_users
conf_for_model["num_bundles"] = dataset.num_bundles
conf_for_model["num_items"] = dataset.num_items
conf_for_model["embedding_size"] = conf_for_model["embedding_sizes"][0]
conf_for_model["num_layers"] = conf_for_model["num_layerss"][0]
conf_for_model["l2_reg"] = conf_for_model["l2_regs"][0]
conf_for_model["c_temp"] = conf_for_model["c_temps"][0]
conf_for_model["item_level_ratio"] = conf_for_model["item_level_ratios"][0]
conf_for_model["bundle_level_ratio"] = conf_for_model["bundle_level_ratios"][0]
conf_for_model["bundle_agg_ratio"] = conf_for_model["bundle_agg_ratios"][0]

model = PAL(conf_for_model, dataset.graphs).to(conf_for_model["device"])
print(model.__class__.__name__, "initialized successfully")

batch = next(iter(dataset.train_loader))
batch = [x.to(conf_for_model["device"]) for x in batch]
bpr_loss, c_loss = model(batch, ED_drop=False)

print("bpr_loss =", float(bpr_loss))
print("c_loss   =", float(c_loss))

Imports OK
>>>>>>>>>>B-I statistics>>>>>>>>>>
Average interactions 5.757723808288574
Non-zero rows 0.9967479674796748
Non-zero columns 1.0
Matrix density 0.002042470229597649
>>>>>>>>>>U-I statistics>>>>>>>>>>
Average interactions 30.47064208984375
Non-zero rows 1.0
Non-zero columns 0.5001773678609436
Matrix density 0.010809025126048253
>>>>>>>>>>U-B statistics in train>>>>>>>>>>
Average interactions 1.7729297876358032
Non-zero rows 0.8142673955591551
Non-zero columns 0.3382113821138211
Matrix density 0.0028828125900210205
>>>>>>>>>>U-B statistics in tune>>>>>>>>>>
Average interactions 0.5909765958786011
Non-zero rows 0.41843828035364783
Non-zero columns 0.27479674796747966
Matrix density 0.0009609375300070068
>>>>>>>>>>U-B statistics in test>>>>>>>>>>
Average interactions 0.5909765958786011
Non-zero rows 0.4189782007153945
Non-zero columns 0.2780487804878049
Matrix density 0.0009609375300070068
num_users: 29634
num_bundles: 615
num_items: 2819
train batches: 820
val batches: 232
test 

/tmp/ipykernel_1173/1926821644.py:40: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print("bpr_loss =", float(bpr_loss))


## Define notebook training and evaluation utilities

In [8]:
import csv
import json
import pandas as pd
from tqdm import tqdm
import torch
import torch.optim as optim

METRIC_NAMES = ["recall", "precision", "ndcg", "hit_rate", "map", "mrr", "f1"]

def get_topk_metrics(grd, is_hit, topk):
    epsilon = 1e-8
    device = grd.device

    num_pos = grd.sum(dim=1)
    valid_user = num_pos > 0
    denorm = valid_user.sum().item()

    if denorm == 0:
        return {m: [0, 0] for m in METRIC_NAMES}

    is_hit = is_hit.float()
    hit_cnt = is_hit.sum(dim=1)

    recall = hit_cnt / (num_pos + epsilon)
    precision = hit_cnt / topk
    hit_rate = (hit_cnt > 0).float()
    f1 = 2 * precision * recall / (precision + recall + epsilon)

    rank_positions = torch.arange(1, topk + 1, device=device, dtype=torch.float)
    discounts = torch.log2(rank_positions + 1)

    dcg = (is_hit / discounts).sum(dim=1)
    num_pos_at_k = num_pos.clamp(0, topk).to(torch.long)
    ideal_hits = torch.arange(topk, device=device).unsqueeze(0) < num_pos_at_k.unsqueeze(1)
    idcg = (ideal_hits.float() / discounts).sum(dim=1)
    ndcg = dcg / (idcg + epsilon)

    precision_at_rank = is_hit.cumsum(dim=1) / rank_positions
    ap = (precision_at_rank * is_hit).sum(dim=1) / torch.minimum(
        num_pos,
        torch.tensor(topk, device=device, dtype=num_pos.dtype)
    ).clamp(min=1)

    reciprocal_rank = (is_hit / rank_positions).max(dim=1).values

    return {
        "recall": [recall[valid_user].sum().item(), denorm],
        "precision": [precision[valid_user].sum().item(), denorm],
        "ndcg": [ndcg[valid_user].sum().item(), denorm],
        "hit_rate": [hit_rate[valid_user].sum().item(), denorm],
        "map": [ap[valid_user].sum().item(), denorm],
        "mrr": [reciprocal_rank[valid_user].sum().item(), denorm],
        "f1": [f1[valid_user].sum().item(), denorm],
    }


def get_metrics(metrics, grd, pred, topks):
    grd = grd.to(pred.device)

    for topk in topks:
        _, col_indice = torch.topk(pred, topk)
        row_indice = torch.zeros_like(col_indice) + torch.arange(
            pred.shape[0], device=pred.device, dtype=torch.long
        ).view(-1, 1)

        is_hit = grd[row_indice.view(-1), col_indice.view(-1)].view(-1, topk)
        tmp_topk = get_topk_metrics(grd, is_hit, topk)

        for m in METRIC_NAMES:
            metrics[m][topk][0] += tmp_topk[m][0]
            metrics[m][topk][1] += tmp_topk[m][1]

    return metrics


def evaluate_model(model, dataloader, conf):
    device = conf["device"]
    tmp_metrics = {m: {k: [0, 0] for k in conf["topk"]} for m in METRIC_NAMES}

    model.eval()
    with torch.no_grad():
        rs = model.propagate(test=True)
        for users, ground_truth_u_b, train_mask_u_b in dataloader:
            pred_b = model.evaluate(rs, users.to(device))
            pred_b -= 1e8 * train_mask_u_b.to(device)
            tmp_metrics = get_metrics(tmp_metrics, ground_truth_u_b, pred_b, conf["topk"])

    metrics = {
        m: {
            k: (tmp_metrics[m][k][0] / tmp_metrics[m][k][1] if tmp_metrics[m][k][1] else 0.0)
            for k in conf["topk"]
        }
        for m in METRIC_NAMES
    }
    return metrics


def print_metrics_table(metrics, split_name):
    print(f"\n[{split_name}]")
    for metric_name in METRIC_NAMES:
        line = "  ".join([f"{metric_name}@{k}={metrics[metric_name][k]:.6f}" for k in metrics[metric_name]])
        print(line)


def flatten_metrics_for_row(prefix, metrics):
    row = {}
    for metric_name in METRIC_NAMES:
        for k, v in metrics[metric_name].items():
            row[f"{prefix}_{metric_name}@{k}"] = v
    return row


def train_notebook(conf, experiment_name="colab_run"):
    device = conf["device"]

    dataset = Datasets(conf)

    conf = dict(conf)
    conf["num_users"] = dataset.num_users
    conf["num_bundles"] = dataset.num_bundles
    conf["num_items"] = dataset.num_items

    conf["embedding_size"] = conf["embedding_sizes"][0]
    conf["num_layers"] = conf["num_layerss"][0]
    conf["l2_reg"] = conf["l2_regs"][0]
    conf["c_lambda"] = conf["c_lambdas"][0]
    conf["c_temp"] = conf["c_temps"][0]
    conf["item_level_ratio"] = conf["item_level_ratios"][0]
    conf["bundle_level_ratio"] = conf["bundle_level_ratios"][0]
    conf["bundle_agg_ratio"] = conf["bundle_agg_ratios"][0]
    lr = conf["lrs"][0]

    model = PAL(conf, dataset.graphs).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=conf["l2_reg"])

    ckpt_dir = PROJECT_ROOT / "checkpoints" / conf["dataset"] / conf["model"] / "model"
    conf_dir = PROJECT_ROOT / "checkpoints" / conf["dataset"] / conf["model"] / "conf"
    result_dir = PROJECT_ROOT / "results"

    ckpt_dir.mkdir(parents=True, exist_ok=True)
    conf_dir.mkdir(parents=True, exist_ok=True)
    result_dir.mkdir(parents=True, exist_ok=True)

    checkpoint_model_path = ckpt_dir / experiment_name
    checkpoint_conf_path = conf_dir / experiment_name
    result_csv = result_dir / f"{conf['dataset']}_{conf['model']}_{experiment_name}_results.csv"

    best_val_recall20 = -1.0
    best_val_ndcg20 = -1.0
    best_epoch = None
    history = []

    for epoch in range(conf["epochs"]):
        model.train()
        pbar = tqdm(dataset.train_loader, desc=f"Epoch {epoch+1}/{conf['epochs']}")

        for batch in pbar:
            batch = [x.to(device) for x in batch]

            optimizer.zero_grad()
            bpr_loss, c_loss = model(batch, ED_drop=False)
            loss = bpr_loss + conf["c_lambda"] * c_loss
            loss.backward()
            optimizer.step()

            pbar.set_postfix(
                loss=float(loss.detach().cpu()),
                bpr=float(bpr_loss.detach().cpu()),
                c=float(c_loss.detach().cpu())
            )

        val_metrics = evaluate_model(model, dataset.val_loader, conf)
        test_metrics = evaluate_model(model, dataset.test_loader, conf)

        print_metrics_table(val_metrics, "VAL")
        print_metrics_table(test_metrics, "TEST")

        row = {"epoch": epoch + 1}
        row.update(flatten_metrics_for_row("val", val_metrics))
        row.update(flatten_metrics_for_row("test", test_metrics))
        history.append(row)

        is_best = (
            val_metrics["recall"].get(20, -1) > best_val_recall20
            and val_metrics["ndcg"].get(20, -1) > best_val_ndcg20
        )

        if is_best:
            best_val_recall20 = val_metrics["recall"][20]
            best_val_ndcg20 = val_metrics["ndcg"][20]
            best_epoch = epoch + 1

            torch.save(model.state_dict(), checkpoint_model_path)

            dump_conf = dict(conf)
            dump_conf["device"] = str(conf["device"])
            with open(checkpoint_conf_path, "w", encoding="utf-8") as f:
                json.dump(dump_conf, f, ensure_ascii=False, indent=2)

            print(f"\nSaved new best checkpoint at epoch {best_epoch}")

    history_df = pd.DataFrame(history)
    history_df.to_csv(result_csv, index=False)

    return {
        "model": model,
        "history": history,
        "history_df": history_df,
        "best_epoch": best_epoch,
        "checkpoint_model_path": str(checkpoint_model_path),
        "checkpoint_conf_path": str(checkpoint_conf_path),
        "result_csv": str(result_csv),
    }

## Define experiment configuration helper

In [9]:
def make_experiment_conf(base_conf, *, attention_type=None, fusion_type=None, epochs=100):
    conf = dict(base_conf)
    conf["epochs"] = epochs

    # attention
    if attention_type == "none":
        conf["use_item_attention"] = False
    elif attention_type in {"global", "user"}:
        conf["use_item_attention"] = True
        conf["attention_type"] = attention_type

    # fusion
    if fusion_type == "none":
        conf["use_view_fusion"] = False
    elif fusion_type in {"global", "user"}:
        conf["use_view_fusion"] = True
        conf["fusion_type"] = fusion_type

    return conf

## Run baseline experiment

In [10]:
baseline_conf = make_experiment_conf(
    base_conf,
    attention_type="none",
    fusion_type="none",
    epochs=100,
)

baseline_output = train_notebook(
    baseline_conf,
    experiment_name="step0_baseline"
)

baseline_output["history_df"]

>>>>>>>>>>B-I statistics>>>>>>>>>>
Average interactions 5.757723808288574
Non-zero rows 0.9967479674796748
Non-zero columns 1.0
Matrix density 0.002042470229597649
>>>>>>>>>>U-I statistics>>>>>>>>>>
Average interactions 30.47064208984375
Non-zero rows 1.0
Non-zero columns 0.5001773678609436
Matrix density 0.010809025126048253
>>>>>>>>>>U-B statistics in train>>>>>>>>>>
Average interactions 1.7729297876358032
Non-zero rows 0.8142673955591551
Non-zero columns 0.3382113821138211
Matrix density 0.0028828125900210205
>>>>>>>>>>U-B statistics in tune>>>>>>>>>>
Average interactions 0.5909765958786011
Non-zero rows 0.41843828035364783
Non-zero columns 0.27479674796747966
Matrix density 0.0009609375300070068
>>>>>>>>>>U-B statistics in test>>>>>>>>>>
Average interactions 0.5909765958786011
Non-zero rows 0.4189782007153945
Non-zero columns 0.2780487804878049
Matrix density 0.0009609375300070068


Epoch 1/100: 100%|██████████| 820/820 [00:11<00:00, 70.78it/s, bpr=0.123, c=3.32, loss=0.256]



[VAL]
recall@1=0.335721  recall@2=0.478756  recall@5=0.817926  recall@10=0.941038  recall@20=0.977361
precision@1=0.434113  precision@2=0.328669  precision@5=0.222935  precision@10=0.129976  precision@20=0.067968
ndcg@1=0.434113  ndcg@2=0.472380  ndcg@5=0.616419  ndcg@10=0.661223  ndcg@20=0.671713
hit_rate@1=0.434113  hit_rate@2=0.557339  hit_rate@5=0.867823  hit_rate@10=0.961613  hit_rate@20=0.986935
map@1=0.434113  map@2=0.447440  map@5=0.535190  map@10=0.559088  map@20=0.563063
mrr@1=0.434113  mrr@2=0.495726  mrr@5=0.573700  mrr@10=0.587159  mrr@20=0.589060
f1@1=0.364598  f1@2=0.372641  f1@5=0.337584  f1@10=0.222291  f1@20=0.124896

[TEST]
recall@1=0.332480  recall@2=0.482460  recall@5=0.820265  recall@10=0.942907  recall@20=0.979150
precision@1=0.431540  precision@2=0.332756  precision@5=0.223454  precision@10=0.130074  precision@20=0.068065
ndcg@1=0.431540  ndcg@2=0.474672  ndcg@5=0.617261  ndcg@10=0.661662  ndcg@20=0.672155
hit_rate@1=0.431540  hit_rate@2=0.560567  hit_rate@5=0.

Epoch 2/100: 100%|██████████| 820/820 [00:10<00:00, 74.58it/s, bpr=0.0773, c=3.06, loss=0.2]



[VAL]
recall@1=0.390247  recall@2=0.546512  recall@5=0.850609  recall@10=0.949690  recall@20=0.979880
precision@1=0.493629  precision@2=0.368427  precision@5=0.231968  precision@10=0.131056  precision@20=0.068206
ndcg@1=0.493629  ndcg@2=0.538163  ndcg@5=0.667255  ndcg@10=0.703182  ndcg@20=0.712071
hit_rate@1=0.493629  hit_rate@2=0.628952  hit_rate@5=0.894839  hit_rate@10=0.969194  hit_rate@20=0.988468
map@1=0.493629  map@2=0.511472  map@5=0.591982  map@10=0.611324  map@20=0.614832
mrr@1=0.493629  mrr@2=0.561290  mrr@5=0.629133  mrr@10=0.639629  mrr@20=0.641077
f1@1=0.420623  f1@2=0.421255  f1@5=0.351167  f1@10=0.224191  f1@20=0.125304

[TEST]
recall@1=0.380225  recall@2=0.540955  recall@5=0.852981  recall@10=0.949890  recall@20=0.980958
precision@1=0.483086  precision@2=0.365697  precision@5=0.232539  precision@10=0.131153  precision@20=0.068226
ndcg@1=0.483086  ndcg@2=0.530780  ndcg@5=0.664158  ndcg@10=0.699425  ndcg@20=0.708469
hit_rate@1=0.483086  hit_rate@2=0.623792  hit_rate@5=0.

...

Epoch 99/100: 100%|██████████| 820/820 [00:11<00:00, 73.65it/s, bpr=0.0356, c=2.73, loss=0.145]



[VAL]
recall@1=0.527821  recall@2=0.732727  recall@5=0.922012  recall@10=0.964469  recall@20=0.985190
precision@1=0.638387  precision@2=0.471774  precision@5=0.250710  precision@10=0.133185  precision@20=0.068593
ndcg@1=0.638387  ndcg@2=0.710631  ndcg@5=0.791133  ndcg@10=0.807398  ndcg@20=0.813531
hit_rate@1=0.638387  hit_rate@2=0.816694  hit_rate@5=0.954355  hit_rate@10=0.979758  hit_rate@20=0.992581
map@1=0.638387  map@2=0.678911  map@5=0.734108  map@10=0.744043  map@20=0.746480
mrr@1=0.638387  mrr@2=0.727540  mrr@5=0.764495  mrr@10=0.768163  mrr@20=0.769099
f1@1=0.560391  f1@2=0.551889  f1@5=0.379876  f1@10=0.227802  f1@20=0.126005

[TEST]
recall@1=0.530695  recall@2=0.737758  recall@5=0.924243  recall@10=0.967116  recall@20=0.985538
precision@1=0.642316  precision@2=0.475314  precision@5=0.251546  precision@10=0.133473  precision@20=0.068549
ndcg@1=0.642316  ndcg@2=0.715413  ndcg@5=0.794819  ndcg@10=0.810912  ndcg@20=0.816385
hit_rate@1=0.642316  hit_rate@2=0.820151  hit_rate@5=0.

Epoch 100/100: 100%|██████████| 820/820 [00:11<00:00, 73.09it/s, bpr=0.0284, c=2.66, loss=0.135]



[VAL]
recall@1=0.525813  recall@2=0.728932  recall@5=0.917700  recall@10=0.963634  recall@20=0.984658
precision@1=0.636613  precision@2=0.468831  precision@5=0.249548  precision@10=0.133153  precision@20=0.068548
ndcg@1=0.636613  ndcg@2=0.707196  ndcg@5=0.787251  ndcg@10=0.804717  ndcg@20=0.810904
hit_rate@1=0.636613  hit_rate@2=0.812823  hit_rate@5=0.950887  hit_rate@10=0.978710  hit_rate@20=0.991935
map@1=0.636613  map@2=0.675484  map@5=0.730042  map@10=0.740583  map@20=0.743035
mrr@1=0.636613  mrr@2=0.724718  mrr@5=0.761487  mrr@10=0.765470  mrr@20=0.766453
f1@1=0.558462  f1@2=0.548669  f1@5=0.378058  f1@10=0.227683  f1@20=0.125919

[TEST]
recall@1=0.524650  recall@2=0.730813  recall@5=0.919472  recall@10=0.965324  recall@20=0.985728
precision@1=0.636276  precision@2=0.471368  precision@5=0.250113  precision@10=0.133304  precision@20=0.068549
ndcg@1=0.636276  ndcg@2=0.708633  ndcg@5=0.788790  ndcg@10=0.806059  ndcg@20=0.812015
hit_rate@1=0.636276  hit_rate@2=0.813628  hit_rate@5=0.

,epoch,val_recall@1,val_recall@2,val_recall@5,val_recall@10,val_recall@20,val_precision@1,val_precision@2,val_precision@5,val_precision@10,...,test_mrr@1,test_mrr@2,test_mrr@5,test_mrr@10,test_mrr@20,test_f1@1,test_f1@2,test_f1@5,test_f1@10,test_f1@20
0,1,0.335721,0.478756,0.817926,0.941038,0.977361,0.434113,0.328669,0.222935,0.129976,...,0.431540,0.496053,0.573535,0.586829,0.588606,0.361516,0.376378,0.338593,0.222602,0.125101
1,2,0.390247,0.546512,0.850609,0.949690,0.979880,0.493629,0.368427,0.231968,0.131056,...,0.483086,0.553439,0.623484,0.633702,0.635284,0.410456,0.417765,0.352196,0.224401,0.125372
2,3,0.404761,0.562623,0.857423,0.953509,0.982531,0.508952,0.376411,0.234048,0.131629,...,0.498470,0.572688,0.638599,0.648392,0.649803,0.424901,0.433591,0.355287,0.225359,0.125531
3,4,0.462694,0.633195,0.875433,0.956241,0.982363,0.570242,0.416008,0.239177,0.132073,...,0.558231,0.639054,0.690750,0.699194,0.700408,0.482398,0.483491,0.362461,0.226336,0.125681
4,5,0.477018,0.654747,0.883318,0.957218,0.982886,0.585161,0.428145,0.241032,0.132169,...,0.579736,0.660519,0.708470,0.716102,0.717327,0.503921,0.497870,0.365358,0.226384,0.125728
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,96,0.517509,0.718418,0.916286,0.963198,0.985607,0.626935,0.462944,0.248823,0.133008,...,0.634987,0.719515,0.758511,0.762563,0.763526,0.556598,0.543473,0.378577,0.227780,0.125990
96,97,0.514399,0.710614,0.916216,0.962058,0.985553,0.622742,0.458831,0.249000,0.132831,...,0.627416,0.711985,0.752306,0.756628,0.757584,0.548998,0.539679,0.378182,0.227769,0.126015
97,98,0.525817,0.724765,0.918105,0.963501,0.985116,0.636290,0.467097,0.249645,0.133073,...,0.632974,0.721045,0.759292,0.763171,0.764131,0.555679,0.548171,0.379461,0.228118,0.126007
98,99,0.527821,0.732727,0.922012,0.964469,0.985190,0.638387,0.471774,0.250710,0.133185,...,0.642316,0.731234,0.767664,0.771305,0.772123,0.563599,0.555957,0.381234,0.228416,0.125977


## Run global attention experiment

In [11]:
global_attention_conf = make_experiment_conf(
    base_conf,
    attention_type="global",
    fusion_type="none",
    epochs=100,
)

global_attention_output = train_notebook(
    global_attention_conf,
    experiment_name="step1a_global_attention"
)

global_attention_output["history_df"]

>>>>>>>>>>B-I statistics>>>>>>>>>>
Average interactions 5.757723808288574
Non-zero rows 0.9967479674796748
Non-zero columns 1.0
Matrix density 0.002042470229597649
>>>>>>>>>>U-I statistics>>>>>>>>>>
Average interactions 30.47064208984375
Non-zero rows 1.0
Non-zero columns 0.5001773678609436
Matrix density 0.010809025126048253
>>>>>>>>>>U-B statistics in train>>>>>>>>>>
Average interactions 1.7729297876358032
Non-zero rows 0.8142673955591551
Non-zero columns 0.3382113821138211
Matrix density 0.0028828125900210205
>>>>>>>>>>U-B statistics in tune>>>>>>>>>>
Average interactions 0.5909765958786011
Non-zero rows 0.41843828035364783
Non-zero columns 0.27479674796747966
Matrix density 0.0009609375300070068
>>>>>>>>>>U-B statistics in test>>>>>>>>>>
Average interactions 0.5909765958786011
Non-zero rows 0.4189782007153945
Non-zero columns 0.2780487804878049
Matrix density 0.0009609375300070068


Epoch 1/100: 100%|██████████| 820/820 [00:21<00:00, 37.41it/s, bpr=0.119, c=3.42, loss=0.256]



[VAL]
recall@1=0.393726  recall@2=0.559011  recall@5=0.805930  recall@10=0.939555  recall@20=0.976257
precision@1=0.496855  precision@2=0.371048  precision@5=0.219032  precision@10=0.129629  precision@20=0.067883
ndcg@1=0.496855  ndcg@2=0.545739  ndcg@5=0.648289  ndcg@10=0.696659  ndcg@20=0.707318
hit_rate@1=0.496855  hit_rate@2=0.646774  hit_rate@5=0.859194  hit_rate@10=0.960000  hit_rate@20=0.986210
map@1=0.496855  map@2=0.515746  map@5=0.578940  map@10=0.604638  map@20=0.608676
mrr@1=0.496855  mrr@2=0.571815  mrr@5=0.624792  mrr@10=0.638988  mrr@20=0.640950
f1@1=0.424071  f1@2=0.427527  f1@5=0.331969  f1@10=0.221799  f1@20=0.124747

[TEST]
recall@1=0.388874  recall@2=0.559708  recall@5=0.807011  recall@10=0.939672  recall@20=0.977304
precision@1=0.491382  precision@2=0.373228  precision@5=0.219749  precision@10=0.129679  precision@20=0.067856
ndcg@1=0.491382  ndcg@2=0.545220  ndcg@5=0.647428  ndcg@10=0.695154  ndcg@20=0.706002
hit_rate@1=0.491382  hit_rate@2=0.647229  hit_rate@5=0.

Epoch 2/100: 100%|██████████| 820/820 [00:21<00:00, 37.39it/s, bpr=0.0579, c=3.18, loss=0.185]



[VAL]
recall@1=0.480969  recall@2=0.653415  recall@5=0.858813  recall@10=0.950231  recall@20=0.979047
precision@1=0.591129  precision@2=0.426290  precision@5=0.234387  precision@10=0.131129  precision@20=0.068137
ndcg@1=0.591129  ndcg@2=0.641213  ndcg@5=0.727277  ndcg@10=0.760713  ndcg@20=0.769210
hit_rate@1=0.591129  hit_rate@2=0.741210  hit_rate@5=0.901935  hit_rate@10=0.969435  hit_rate@20=0.988065
map@1=0.591129  map@2=0.611552  map@5=0.667694  map@10=0.685922  map@20=0.689284
mrr@1=0.591129  mrr@2=0.666169  mrr@5=0.707860  mrr@10=0.717634  mrr@20=0.719044
f1@1=0.513454  f1@2=0.495407  f1@5=0.354760  f1@10=0.224302  f1@20=0.125186

[TEST]
recall@1=0.476981  recall@2=0.655029  recall@5=0.860164  recall@10=0.950822  recall@20=0.980329
precision@1=0.587065  precision@2=0.428721  precision@5=0.234584  precision@10=0.131322  precision@20=0.068146
ndcg@1=0.587065  ndcg@2=0.641443  ndcg@5=0.726877  ndcg@10=0.759990  ndcg@20=0.768536
hit_rate@1=0.587065  hit_rate@2=0.743396  hit_rate@5=0.

...

Epoch 99/100: 100%|██████████| 820/820 [00:22<00:00, 37.12it/s, bpr=0.0363, c=2.62, loss=0.141]



[VAL]
recall@1=0.539166  recall@2=0.739532  recall@5=0.921223  recall@10=0.960055  recall@20=0.984540
precision@1=0.649758  precision@2=0.472218  precision@5=0.250177  precision@10=0.132677  precision@20=0.068621
ndcg@1=0.649758  ndcg@2=0.717603  ndcg@5=0.797977  ndcg@10=0.812996  ndcg@20=0.820222
hit_rate@1=0.649758  hit_rate@2=0.824919  hit_rate@5=0.952661  hit_rate@10=0.976048  hit_rate@20=0.991694
map@1=0.649758  map@2=0.685383  map@5=0.742639  map@10=0.751884  map@20=0.754716
mrr@1=0.649758  mrr@2=0.737339  mrr@5=0.774507  mrr@10=0.777830  mrr@20=0.778997
f1@1=0.571910  f1@2=0.554825  f1@5=0.379322  f1@10=0.226851  f1@20=0.126029

[TEST]
recall@1=0.541551  recall@2=0.733449  recall@5=0.921653  recall@10=0.961787  recall@20=0.985481
precision@1=0.652626  precision@2=0.469354  precision@5=0.250789  precision@10=0.132780  precision@20=0.068529
ndcg@1=0.652626  ndcg@2=0.715088  ndcg@5=0.798805  ndcg@10=0.813944  ndcg@20=0.820824
hit_rate@1=0.652626  hit_rate@2=0.818702  hit_rate@5=0.

Epoch 100/100: 100%|██████████| 820/820 [00:21<00:00, 37.33it/s, bpr=0.042, c=2.55, loss=0.144]



[VAL]
recall@1=0.539684  recall@2=0.740385  recall@5=0.919931  recall@10=0.959561  recall@20=0.984956
precision@1=0.650323  precision@2=0.471855  precision@5=0.249935  precision@10=0.132589  precision@20=0.068569
ndcg@1=0.650323  ndcg@2=0.718179  ndcg@5=0.797883  ndcg@10=0.813130  ndcg@20=0.820510
hit_rate@1=0.650323  hit_rate@2=0.827500  hit_rate@5=0.951694  hit_rate@10=0.975806  hit_rate@20=0.992581
map@1=0.650323  map@2=0.685363  map@5=0.742670  map@10=0.751992  map@20=0.754831
mrr@1=0.650323  mrr@2=0.738911  mrr@5=0.775253  mrr@10=0.778686  mrr@20=0.779900
f1@1=0.572339  f1@2=0.554872  f1@5=0.378913  f1@10=0.226713  f1@20=0.125960

[TEST]
recall@1=0.542855  recall@2=0.739221  recall@5=0.921461  recall@10=0.960382  recall@20=0.986032
precision@1=0.654720  precision@2=0.472737  precision@5=0.250822  precision@10=0.132700  precision@20=0.068593
ndcg@1=0.654720  ndcg@2=0.719431  ndcg@5=0.800238  ndcg@10=0.814986  ndcg@20=0.822336
hit_rate@1=0.654720  hit_rate@2=0.824259  hit_rate@5=0.

,epoch,val_recall@1,val_recall@2,val_recall@5,val_recall@10,val_recall@20,val_precision@1,val_precision@2,val_precision@5,val_precision@10,...,test_mrr@1,test_mrr@2,test_mrr@5,test_mrr@10,test_mrr@20,test_f1@1,test_f1@2,test_f1@5,test_f1@10,test_f1@20
0,1,0.393726,0.559011,0.805930,0.939555,0.976257,0.496855,0.371048,0.219032,0.129629,...,0.491382,0.569306,0.622019,0.636170,0.638153,0.418937,0.429004,0.332936,0.221910,0.124761
1,2,0.480969,0.653415,0.858813,0.950231,0.979047,0.591129,0.426290,0.234387,0.131129,...,0.587065,0.665230,0.706151,0.715707,0.717190,0.509439,0.497445,0.355261,0.224646,0.125245
2,3,0.536070,0.717269,0.888246,0.953289,0.980303,0.647984,0.461613,0.242016,0.131613,...,0.641994,0.722616,0.756308,0.763048,0.764254,0.562269,0.538752,0.367315,0.225716,0.125595
3,4,0.555337,0.741309,0.905640,0.957361,0.982297,0.667258,0.474435,0.246548,0.132177,...,0.654317,0.740174,0.773152,0.778091,0.779136,0.576136,0.555459,0.374062,0.226689,0.125730
4,5,0.556700,0.747001,0.914106,0.958971,0.981927,0.668306,0.476613,0.248726,0.132403,...,0.654317,0.742469,0.776278,0.780551,0.781588,0.576666,0.559284,0.377345,0.226818,0.125663
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,96,0.534860,0.736486,0.918046,0.957918,0.984551,0.645242,0.470403,0.249532,0.132347,...,0.654075,0.738080,0.775236,0.778714,0.779978,0.574616,0.552610,0.378918,0.226914,0.126008
96,97,0.539688,0.742748,0.919877,0.960477,0.985636,0.650161,0.473629,0.249968,0.132581,...,0.655606,0.741503,0.777865,0.781194,0.782412,0.576475,0.556248,0.379918,0.227241,0.125969
97,98,0.540717,0.734460,0.916650,0.959031,0.984684,0.652258,0.469435,0.248952,0.132532,...,0.649887,0.733449,0.772366,0.775944,0.777254,0.571585,0.549519,0.378173,0.226737,0.125979
98,99,0.539166,0.739532,0.921223,0.960055,0.984540,0.649758,0.472218,0.250177,0.132677,...,0.652626,0.735664,0.774929,0.778350,0.779476,0.574376,0.550708,0.380126,0.227192,0.125925


## Run user-aware attention experiment

In [12]:
user_attention_conf = make_experiment_conf(
    base_conf,
    attention_type="user",
    fusion_type="none",
    epochs=100,
)

user_attention_output = train_notebook(
    user_attention_conf,
    experiment_name="step1b_user_attention"
)

user_attention_output["history_df"]

>>>>>>>>>>B-I statistics>>>>>>>>>>
Average interactions 5.757723808288574
Non-zero rows 0.9967479674796748
Non-zero columns 1.0
Matrix density 0.002042470229597649
>>>>>>>>>>U-I statistics>>>>>>>>>>
Average interactions 30.47064208984375
Non-zero rows 1.0
Non-zero columns 0.5001773678609436
Matrix density 0.010809025126048253
>>>>>>>>>>U-B statistics in train>>>>>>>>>>
Average interactions 1.7729297876358032
Non-zero rows 0.8142673955591551
Non-zero columns 0.3382113821138211
Matrix density 0.0028828125900210205
>>>>>>>>>>U-B statistics in tune>>>>>>>>>>
Average interactions 0.5909765958786011
Non-zero rows 0.41843828035364783
Non-zero columns 0.27479674796747966
Matrix density 0.0009609375300070068
>>>>>>>>>>U-B statistics in test>>>>>>>>>>
Average interactions 0.5909765958786011
Non-zero rows 0.4189782007153945
Non-zero columns 0.2780487804878049
Matrix density 0.0009609375300070068


Epoch 1/100: 100%|██████████| 820/820 [00:13<00:00, 60.96it/s, bpr=0.0983, c=3.35, loss=0.232]



[VAL]
recall@1=0.348762  recall@2=0.499190  recall@5=0.790048  recall@10=0.936783  recall@20=0.978517
precision@1=0.445887  precision@2=0.338145  precision@5=0.216613  precision@10=0.129298  precision@20=0.067992
ndcg@1=0.445887  ndcg@2=0.490003  ndcg@5=0.613449  ndcg@10=0.666876  ndcg@20=0.678915
hit_rate@1=0.445887  hit_rate@2=0.584113  hit_rate@5=0.841210  hit_rate@10=0.958387  hit_rate@20=0.987581
map@1=0.445887  map@2=0.462500  map@5=0.539021  map@10=0.567069  map@20=0.571647
mrr@1=0.445887  mrr@2=0.515000  mrr@5=0.580539  mrr@10=0.598033  mrr@20=0.600241
f1@1=0.377137  f1@2=0.385593  f1@5=0.327362  f1@10=0.221190  f1@20=0.124977

[TEST]
recall@1=0.344554  recall@2=0.496969  recall@5=0.791324  recall@10=0.939084  recall@20=0.979692
precision@1=0.441688  precision@2=0.337307  precision@5=0.217091  precision@10=0.129663  precision@20=0.068082
ndcg@1=0.441688  ndcg@2=0.487095  ndcg@5=0.612088  ndcg@10=0.665560  ndcg@20=0.677223
hit_rate@1=0.441688  hit_rate@2=0.580380  hit_rate@5=0.

Epoch 2/100: 100%|██████████| 820/820 [00:13<00:00, 60.41it/s, bpr=0.0757, c=3.08, loss=0.199]



[VAL]
recall@1=0.442059  recall@2=0.620403  recall@5=0.823707  recall@10=0.941959  recall@20=0.980409
precision@1=0.544597  precision@2=0.404395  precision@5=0.225565  precision@10=0.130202  precision@20=0.068246
ndcg@1=0.544597  ndcg@2=0.603283  ndcg@5=0.692006  ndcg@10=0.735270  ndcg@20=0.746356
hit_rate@1=0.544597  hit_rate@2=0.711129  hit_rate@5=0.871129  hit_rate@10=0.962258  hit_rate@20=0.989194
map@1=0.544597  map@2=0.571492  map@5=0.630821  map@10=0.653988  map@20=0.658186
mrr@1=0.544597  mrr@2=0.627863  mrr@5=0.673132  mrr@10=0.686759  mrr@20=0.688787
f1@1=0.472118  f1@2=0.470018  f1@5=0.341048  f1@10=0.222649  f1@20=0.125372

[TEST]
recall@1=0.441041  recall@2=0.620797  recall@5=0.824275  recall@10=0.942585  recall@20=0.981375
precision@1=0.545506  precision@2=0.404438  precision@5=0.225693  precision@10=0.130292  precision@20=0.068251
ndcg@1=0.545506  ndcg@2=0.602963  ndcg@5=0.692204  ndcg@10=0.735387  ndcg@20=0.746542
hit_rate@1=0.545506  hit_rate@2=0.710535  hit_rate@5=0.

...

Epoch 99/100: 100%|██████████| 820/820 [00:13<00:00, 59.66it/s, bpr=0.0442, c=2.53, loss=0.145]



[VAL]
recall@1=0.456607  recall@2=0.661802  recall@5=0.884386  recall@10=0.943506  recall@20=0.981163
precision@1=0.562177  precision@2=0.426573  precision@5=0.240097  precision@10=0.130508  precision@20=0.068363
ndcg@1=0.562177  ndcg@2=0.635500  ndcg@5=0.735027  ndcg@10=0.757086  ndcg@20=0.767986
hit_rate@1=0.562177  hit_rate@2=0.750806  hit_rate@5=0.925645  hit_rate@10=0.962581  hit_rate@20=0.989274
map@1=0.562177  map@2=0.600665  map@5=0.668549  map@10=0.681499  map@20=0.685638
mrr@1=0.562177  mrr@2=0.656492  mrr@5=0.707473  mrr@10=0.712499  mrr@20=0.714527
f1@1=0.487721  f1@2=0.498565  f1@5=0.363962  f1@10=0.223125  f1@20=0.125569

[TEST]
recall@1=0.458895  recall@2=0.663030  recall@5=0.885760  recall@10=0.945949  recall@20=0.982453
precision@1=0.564433  precision@2=0.428359  precision@5=0.240512  precision@10=0.130799  precision@20=0.068327
ndcg@1=0.564433  ndcg@2=0.637459  ndcg@5=0.736947  ndcg@10=0.759278  ndcg@20=0.769781
hit_rate@1=0.564433  hit_rate@2=0.750966  hit_rate@5=0.

Epoch 100/100: 100%|██████████| 820/820 [00:13<00:00, 60.88it/s, bpr=0.0228, c=2.64, loss=0.128]



[VAL]
recall@1=0.456372  recall@2=0.663456  recall@5=0.880958  recall@10=0.945416  recall@20=0.982192
precision@1=0.560806  precision@2=0.427097  precision@5=0.239097  precision@10=0.130710  precision@20=0.068411
ndcg@1=0.560806  ndcg@2=0.636153  ndcg@5=0.733205  ndcg@10=0.757067  ndcg@20=0.767652
hit_rate@1=0.560806  hit_rate@2=0.752984  hit_rate@5=0.922661  hit_rate@10=0.964435  hit_rate@20=0.990403
map@1=0.560806  map@2=0.600907  map@5=0.667171  map@10=0.680985  map@20=0.684962
mrr@1=0.560806  mrr@2=0.656895  mrr@5=0.706222  mrr@10=0.711829  mrr@20=0.713742
f1@1=0.487246  f1@2=0.499500  f1@5=0.362495  f1@10=0.223484  f1@20=0.125656

[TEST]
recall@1=0.457007  recall@2=0.665210  recall@5=0.883754  recall@10=0.948248  recall@20=0.982950
precision@1=0.562097  precision@2=0.429607  precision@5=0.239997  precision@10=0.131169  precision@20=0.068351
ndcg@1=0.562097  ndcg@2=0.638259  ndcg@5=0.735157  ndcg@10=0.758978  ndcg@20=0.768921
hit_rate@1=0.562097  hit_rate@2=0.753947  hit_rate@5=0.

,epoch,val_recall@1,val_recall@2,val_recall@5,val_recall@10,val_recall@20,val_precision@1,val_precision@2,val_precision@5,val_precision@10,...,test_mrr@1,test_mrr@2,test_mrr@5,test_mrr@10,test_mrr@20,test_f1@1,test_f1@2,test_f1@5,test_f1@10,test_f1@20
0,1,0.348762,0.499190,0.790048,0.936783,0.978517,0.445887,0.338145,0.216613,0.129298,...,0.441688,0.511034,0.577141,0.594627,0.596754,0.372991,0.384377,0.328272,0.221858,0.125136
1,2,0.442059,0.620403,0.823707,0.941959,0.980409,0.544597,0.404395,0.225565,0.130202,...,0.545506,0.628020,0.673207,0.686737,0.688803,0.471866,0.470422,0.341374,0.222837,0.125421
2,3,0.439178,0.665954,0.864406,0.946239,0.981651,0.537581,0.427339,0.234645,0.130790,...,0.544378,0.650209,0.692767,0.701262,0.703017,0.472079,0.501842,0.355127,0.224234,0.125615
3,4,0.450093,0.683103,0.869028,0.945203,0.981151,0.549516,0.438589,0.235919,0.130669,...,0.552916,0.664747,0.702767,0.710489,0.712244,0.481024,0.517224,0.356906,0.224333,0.125607
4,5,0.464018,0.689319,0.879969,0.944731,0.980593,0.570000,0.443992,0.239597,0.130661,...,0.575387,0.676869,0.717151,0.723400,0.725281,0.498658,0.518668,0.364017,0.223852,0.125636
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,96,0.457105,0.659107,0.885114,0.946684,0.981931,0.560968,0.424073,0.240226,0.130944,...,0.562258,0.655525,0.707163,0.712524,0.714313,0.488036,0.498911,0.366003,0.224486,0.125775
96,97,0.454512,0.659757,0.884839,0.946849,0.982078,0.559355,0.424274,0.240210,0.130879,...,0.555412,0.652022,0.703747,0.708963,0.710829,0.482254,0.498276,0.364923,0.224410,0.125770
97,98,0.446659,0.656347,0.877202,0.943702,0.982631,0.550968,0.422661,0.237968,0.130500,...,0.554688,0.648558,0.699464,0.705580,0.707561,0.481046,0.493765,0.361662,0.223795,0.125819
98,99,0.456607,0.661802,0.884386,0.943506,0.981163,0.562177,0.426573,0.240097,0.130508,...,0.564433,0.657700,0.708328,0.713605,0.715576,0.490029,0.500218,0.364812,0.223693,0.125563


## Run full PAL model

In [13]:
full_model_conf = make_experiment_conf(
    base_conf,
    attention_type="user",
    fusion_type="user",
    epochs=100,
)

full_model_output = train_notebook(
    full_model_conf,
    experiment_name="step2_full_model"
)

full_model_output["history_df"]

>>>>>>>>>>B-I statistics>>>>>>>>>>
Average interactions 5.757723808288574
Non-zero rows 0.9967479674796748
Non-zero columns 1.0
Matrix density 0.002042470229597649
>>>>>>>>>>U-I statistics>>>>>>>>>>
Average interactions 30.47064208984375
Non-zero rows 1.0
Non-zero columns 0.5001773678609436
Matrix density 0.010809025126048253
>>>>>>>>>>U-B statistics in train>>>>>>>>>>
Average interactions 1.7729297876358032
Non-zero rows 0.8142673955591551
Non-zero columns 0.3382113821138211
Matrix density 0.0028828125900210205
>>>>>>>>>>U-B statistics in tune>>>>>>>>>>
Average interactions 0.5909765958786011
Non-zero rows 0.41843828035364783
Non-zero columns 0.27479674796747966
Matrix density 0.0009609375300070068
>>>>>>>>>>U-B statistics in test>>>>>>>>>>
Average interactions 0.5909765958786011
Non-zero rows 0.4189782007153945
Non-zero columns 0.2780487804878049
Matrix density 0.0009609375300070068


Epoch 1/100: 100%|██████████| 820/820 [00:14<00:00, 58.27it/s, bpr=0.0904, c=2.85, loss=0.205]



[VAL]
recall@1=0.342584  recall@2=0.543274  recall@5=0.787423  recall@10=0.926022  recall@20=0.979068
precision@1=0.431290  precision@2=0.356290  precision@5=0.213823  precision@10=0.127879  precision@20=0.068069
ndcg@1=0.431290  ndcg@2=0.512910  ndcg@5=0.619999  ndcg@10=0.670202  ndcg@20=0.685495
hit_rate@1=0.431290  hit_rate@2=0.633548  hit_rate@5=0.843387  hit_rate@10=0.949435  hit_rate@20=0.988306
map@1=0.431290  map@2=0.477157  map@5=0.546696  map@10=0.573037  map@20=0.578785
mrr@1=0.431290  mrr@2=0.532500  mrr@5=0.589831  mrr@10=0.604965  mrr@20=0.607962
f1@1=0.368486  f1@2=0.412753  f1@5=0.324088  f1@10=0.218747  f1@20=0.125105

[TEST]
recall@1=0.337319  recall@2=0.536505  recall@5=0.788232  recall@10=0.928056  recall@20=0.979832
precision@1=0.422600  precision@2=0.352771  precision@5=0.213982  precision@10=0.128141  precision@20=0.068017
ndcg@1=0.422600  ndcg@2=0.506112  ndcg@5=0.617203  ndcg@10=0.667801  ndcg@20=0.682594
hit_rate@1=0.422600  hit_rate@2=0.628947  hit_rate@5=0.

Epoch 2/100: 100%|██████████| 820/820 [00:13<00:00, 58.94it/s, bpr=0.0435, c=2.83, loss=0.157]



[VAL]
recall@1=0.386088  recall@2=0.599191  recall@5=0.855508  recall@10=0.939571  recall@20=0.982032
precision@1=0.471452  precision@2=0.380565  precision@5=0.229887  precision@10=0.129645  precision@20=0.068294
ndcg@1=0.471452  ndcg@2=0.562046  ndcg@5=0.680892  ndcg@10=0.712378  ndcg@20=0.724663
hit_rate@1=0.471452  hit_rate@2=0.681855  hit_rate@5=0.903629  hit_rate@10=0.959919  hit_rate@20=0.990403
map@1=0.471452  map@2=0.525887  map@5=0.606255  map@10=0.624246  map@20=0.628876
mrr@1=0.471452  mrr@2=0.576694  mrr@5=0.642020  mrr@10=0.649806  mrr@20=0.652091
f1@1=0.411109  f1@2=0.447946  f1@5=0.349628  f1@10=0.221796  f1@20=0.125505

[TEST]
recall@1=0.377718  recall@2=0.593013  recall@5=0.854872  recall@10=0.941692  recall@20=0.982445
precision@1=0.461260  precision@2=0.376409  precision@5=0.229639  precision@10=0.129913  precision@20=0.068202
ndcg@1=0.461260  ndcg@2=0.553944  ndcg@5=0.676568  ndcg@10=0.708830  ndcg@20=0.720594
hit_rate@1=0.461260  hit_rate@2=0.677835  hit_rate@5=0.

...

Epoch 99/100: 100%|██████████| 820/820 [00:14<00:00, 57.29it/s, bpr=0.0323, c=2.53, loss=0.134]



[VAL]
recall@1=0.496345  recall@2=0.732617  recall@5=0.919153  recall@10=0.960413  recall@20=0.986436
precision@1=0.608790  precision@2=0.469153  precision@5=0.250048  precision@10=0.132645  precision@20=0.068694
ndcg@1=0.608790  ndcg@2=0.698032  ndcg@5=0.780377  ndcg@10=0.795960  ndcg@20=0.803329
hit_rate@1=0.608790  hit_rate@2=0.824032  hit_rate@5=0.950323  hit_rate@10=0.976774  hit_rate@20=0.993145
map@1=0.608790  map@2=0.659597  map@5=0.718670  map@10=0.728030  map@20=0.730675
mrr@1=0.608790  mrr@2=0.716089  mrr@5=0.753108  mrr@10=0.756788  mrr@20=0.757575
f1@1=0.529465  f1@2=0.550208  f1@5=0.378823  f1@10=0.226851  f1@20=0.126212

[TEST]
recall@1=0.497433  recall@2=0.732167  recall@5=0.921670  recall@10=0.962995  recall@20=0.985539
precision@1=0.609617  precision@2=0.469112  precision@5=0.250515  precision@10=0.132917  precision@20=0.068452
ndcg@1=0.609617  ndcg@2=0.698391  ndcg@5=0.782099  ndcg@10=0.797900  ndcg@20=0.804664
hit_rate@1=0.609617  hit_rate@2=0.822487  hit_rate@5=0.

Epoch 100/100: 100%|██████████| 820/820 [00:14<00:00, 57.10it/s, bpr=0.0292, c=2.63, loss=0.134]



[VAL]
recall@1=0.500367  recall@2=0.742304  recall@5=0.921325  recall@10=0.958991  recall@20=0.986818
precision@1=0.611855  precision@2=0.474395  precision@5=0.250516  precision@10=0.132516  precision@20=0.068685
ndcg@1=0.611855  ndcg@2=0.705631  ndcg@5=0.784875  ndcg@10=0.799463  ndcg@20=0.807296
hit_rate@1=0.611855  hit_rate@2=0.831855  hit_rate@5=0.952258  hit_rate@10=0.975000  hit_rate@20=0.993710
map@1=0.611855  map@2=0.667036  map@5=0.724417  map@10=0.733488  map@20=0.736251
mrr@1=0.611855  mrr@2=0.721532  mrr@5=0.757020  mrr@10=0.760485  mrr@20=0.761412
f1@1=0.533281  f1@2=0.556921  f1@5=0.379682  f1@10=0.226612  f1@20=0.126208

[TEST]
recall@1=0.502058  recall@2=0.744782  recall@5=0.923779  recall@10=0.962178  recall@20=0.986007
precision@1=0.614852  precision@2=0.476120  precision@5=0.251111  precision@10=0.132708  precision@20=0.068488
ndcg@1=0.614852  ndcg@2=0.708218  ndcg@5=0.787288  ndcg@10=0.801720  ndcg@20=0.809007
hit_rate@1=0.614852  hit_rate@2=0.832474  hit_rate@5=0.

,epoch,val_recall@1,val_recall@2,val_recall@5,val_recall@10,val_recall@20,val_precision@1,val_precision@2,val_precision@5,val_precision@10,...,test_mrr@1,test_mrr@2,test_mrr@5,test_mrr@10,test_mrr@20,test_f1@1,test_f1@2,test_f1@5,test_f1@10,test_f1@20
0,1,0.342584,0.543274,0.787423,0.926022,0.979068,0.431290,0.356290,0.213823,0.127879,...,0.422600,0.525693,0.584081,0.599378,0.602249,0.362224,0.408263,0.324524,0.219296,0.125046
1,2,0.386088,0.599191,0.855508,0.939571,0.982032,0.471452,0.380565,0.229887,0.129645,...,0.461260,0.569507,0.635989,0.643825,0.646094,0.402320,0.443522,0.349549,0.222377,0.125390
2,3,0.451094,0.675944,0.877964,0.944046,0.982930,0.552823,0.432903,0.237435,0.130339,...,0.548164,0.655122,0.700592,0.706475,0.708464,0.476741,0.507271,0.360087,0.223559,0.125585
3,4,0.470468,0.695941,0.891355,0.948619,0.985479,0.576452,0.445161,0.241468,0.131089,...,0.567091,0.674090,0.717279,0.722231,0.724372,0.494621,0.520546,0.366514,0.224462,0.125775
4,5,0.488718,0.716913,0.902518,0.948135,0.984958,0.596774,0.457944,0.244694,0.130968,...,0.597294,0.702199,0.739405,0.743976,0.745994,0.520424,0.540147,0.370402,0.224624,0.125752
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,96,0.503423,0.741053,0.919366,0.960188,0.987191,0.614435,0.473226,0.250032,0.132831,...,0.612919,0.719958,0.756496,0.759849,0.761011,0.535139,0.554360,0.379621,0.227540,0.125889
96,97,0.498029,0.737692,0.916404,0.958032,0.986117,0.609677,0.471855,0.249065,0.132435,...,0.616221,0.721609,0.756799,0.760662,0.762034,0.536619,0.553911,0.377762,0.227005,0.125814
97,98,0.496746,0.729308,0.921949,0.960545,0.987178,0.608306,0.467056,0.250823,0.132782,...,0.606556,0.712750,0.752250,0.755625,0.756819,0.527776,0.546999,0.380653,0.227640,0.126012
98,99,0.496345,0.732617,0.919153,0.960413,0.986436,0.608790,0.469153,0.250048,0.132645,...,0.609617,0.716374,0.753883,0.757487,0.758994,0.530698,0.550187,0.379948,0.227510,0.125861


## Compare final rows of all experiments

In [14]:
import pandas as pd

def get_last_row(output, experiment_name):
    df = output["history_df"].copy()
    row = df.iloc[-1].to_dict()
    row["experiment"] = experiment_name
    return row

compare_df = pd.DataFrame([
    get_last_row(baseline_output, "step0_baseline"),
    get_last_row(global_attention_output, "step1a_global_attention"),
    get_last_row(user_attention_output, "step1b_user_attention"),
    get_last_row(full_model_output, "step2_full_model"),
])

cols = ["experiment"] + [c for c in compare_df.columns if c != "experiment"]
compare_df = compare_df[cols]
compare_df

,experiment,epoch,val_recall@1,val_recall@2,val_recall@5,val_recall@10,val_recall@20,val_precision@1,val_precision@2,val_precision@5,...,test_mrr@1,test_mrr@2,test_mrr@5,test_mrr@10,test_mrr@20,test_f1@1,test_f1@2,test_f1@5,test_f1@10,test_f1@20
0,step0_baseline,100.0,0.525813,0.728932,0.917700,0.963634,0.984658,0.636613,0.468831,0.249548,...,0.636276,0.724952,0.761939,0.765859,0.766835,0.557572,0.550981,0.379102,0.228081,0.125983
1,step1a_global_attention,100.0,0.539684,0.740385,0.919931,0.959561,0.984956,0.650323,0.471855,0.249935,...,0.654720,0.739489,0.776956,0.780182,0.781387,0.575916,0.554922,0.380108,0.226978,0.126043
2,step1b_user_attention,100.0,0.456372,0.663456,0.880958,0.945416,0.982192,0.560806,0.427097,0.239097,...,0.562097,0.658022,0.707050,0.712597,0.714482,0.487989,0.501684,0.363936,0.224317,0.125616
3,step2_full_model,100.0,0.500367,0.742304,0.921325,0.958991,0.986818,0.611855,0.474395,0.250516,...,0.614852,0.723985,0.759133,0.762129,0.763781,0.535536,0.559153,0.380857,0.227210,0.125928


## Export explanations from the best full-model checkpoint

In [15]:
!python export_explanations.py \
  --checkpoint "/content/drive/MyDrive/BT4222/PAL/checkpoints/SteamDebug/PAL/model/step2_full_model" \
  --conf "/content/drive/MyDrive/BT4222/PAL/checkpoints/SteamDebug/PAL/conf/step2_full_model" \
  --topn 5 \
  --top-items 3 \
  --split test

>>>>>>>>>>B-I statistics>>>>>>>>>>
Average interactions 5.757723808288574
Non-zero rows 0.9967479674796748
Non-zero columns 1.0
Matrix density 0.002042470229597649
>>>>>>>>>>U-I statistics>>>>>>>>>>
Average interactions 30.47064208984375
Non-zero rows 1.0
Non-zero columns 0.5001773678609436
Matrix density 0.010809025126048253
>>>>>>>>>>U-B statistics in train>>>>>>>>>>
Average interactions 1.7729297876358032
Non-zero rows 0.8142673955591551
Non-zero columns 0.3382113821138211
Matrix density 0.0028828125900210205
>>>>>>>>>>U-B statistics in tune>>>>>>>>>>
Average interactions 0.5909765958786011
Non-zero rows 0.41843828035364783
Non-zero columns 0.27479674796747966
Matrix density 0.0009609375300070068
>>>>>>>>>>U-B statistics in test>>>>>>>>>>
Average interactions 0.5909765958786011
Non-zero rows 0.4189782007153945
Non-zero columns 0.2780487804878049
Matrix density 0.0009609375300070068
Saved explanations to results/explanations_step2_full_model.csv
